# Product Taxonomy Exploration — Coverage, Composition & Quality

Analyst/stakeholder view of `magpie_reference.product_taxonomy` + `product_taxonomy_map` results across every extracted category (TH + SG combined). Answers: how much of the business does the taxonomy cover, what does it look like, and are there any completeness signals worth a second look.

**Not a QA gate tool** — for pass/fail hard-gate checks (dual-mapped products, brand mismatches, etc.) see `script/qa_report.sh`. This notebook is descriptive/exploratory, read-only.

## 1. Setup

In [ ]:
!pip install -q google-cloud-bigquery db-dtypes

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated. Next cell creates the BigQuery client.')

In [ ]:
PROJECT = "sincere-hearth-273704"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)
print(f"BigQuery client ready for project {PROJECT}")

## 2. Load shared data

Three queries reused across every section below, so BigQuery is only hit once each:
- `primary_df` — every mapped product joined to its SKU and brand (one row per product)
- `sku_df` — every SKU entry directly from `product_taxonomy` (one row per SKU, not just mapped ones)
- `gmv_df` — GMV coverage per category, joining the (deduped) map directly against `marketshare_universe_niq`'s latest month — NOT via `universe_taxonomy_overlay`, which only has one category's rows so far

In [ ]:
PRIMARY_SQL = """
SELECT
  m.product_id, m.master_table, m.country, m.platform, m.source,
  SAFE_CAST(m.confidence AS FLOAT64) AS confidence, m.brand_mismatch, m.meta_agent AS map_meta_agent,
  t.taxonomy_id, t.brand_id, b.canonical_name AS brand_name, t.canonical_name, t.size, t.pack_count,
  t.is_multi_size, t.is_multi_variant, t.is_bundle, t.meta_agent AS sku_meta_agent, t.created_at
FROM `sincere-hearth-273704.magpie_reference.product_taxonomy_map` m
JOIN `sincere-hearth-273704.magpie_reference.product_taxonomy` t ON m.taxonomy_id = t.taxonomy_id
LEFT JOIN `sincere-hearth-273704.magpie_reference.brand_dict` b ON t.brand_id = b.brand_id
"""
primary_df = client.query(PRIMARY_SQL).to_dataframe()
print("primary_df:", primary_df.shape)

In [ ]:
SKU_SQL = """
SELECT t.taxonomy_id, t.brand_id, b.canonical_name AS brand_name, t.product_line, t.sub_line,
       t.variant, t.canonical_name, t.size, t.pack_count, t.is_multi_size, t.is_multi_variant,
       t.is_bundle, t.meta_agent, t.created_at
FROM `sincere-hearth-273704.magpie_reference.product_taxonomy` t
LEFT JOIN `sincere-hearth-273704.magpie_reference.brand_dict` b ON t.brand_id = b.brand_id
-- product_taxonomy has at least one duplicate taxonomy_id (SKU-040096, confirmed live) — dedup
-- so every downstream section counts each SKU exactly once
QUALIFY ROW_NUMBER() OVER (PARTITION BY t.taxonomy_id ORDER BY t.updated_at DESC) = 1
"""
sku_df = client.query(SKU_SQL).to_dataframe()
print("sku_df:", sku_df.shape)

In [ ]:
GMV_SQL = """
WITH latest AS (
  SELECT MAX(month) AS m FROM `sincere-hearth-273704.magpie.marketshare_universe_niq`
),
dedup_map AS (
  -- product_taxonomy_map has ~1,450 dual-mapped products (QA Gate G1 violation) — dedup the same
  -- way AGENTS.md's Universe Refresh Pattern does, or GMV coverage exceeds 100% for affected categories
  SELECT * FROM `sincere-hearth-273704.magpie_reference.product_taxonomy_map`
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_id, platform, country, master_table
    ORDER BY CASE source WHEN 'LLM' THEN 0 ELSE 1 END, taxonomy_id
  ) = 1
),
mapped AS (
  SELECT u.master_table, SUM(u.gmv_monthly) AS mapped_gmv
  FROM `sincere-hearth-273704.magpie.marketshare_universe_niq` u, latest
  JOIN dedup_map m
    ON m.product_id = u.product_id AND m.platform = u.ecommerce_platform
    AND m.country = u.country AND m.master_table = u.master_table
  WHERE u.month = latest.m
  GROUP BY 1
),
total AS (
  SELECT master_table, SUM(gmv_monthly) AS total_gmv
  FROM `sincere-hearth-273704.magpie.marketshare_universe_niq`, latest
  WHERE month = latest.m
  GROUP BY 1
)
SELECT t.master_table, COALESCE(mapped.mapped_gmv, 0) AS mapped_gmv, t.total_gmv,
       SAFE_DIVIDE(COALESCE(mapped.mapped_gmv, 0), t.total_gmv) AS coverage
FROM total t LEFT JOIN mapped USING(master_table)
"""
gmv_df = client.query(GMV_SQL).to_dataframe()
print("gmv_df:", gmv_df.shape)
print("coverage range:", gmv_df["coverage"].min(), "-", gmv_df["coverage"].max())

## 3. Headline dashboard

The two numbers that matter most: how many categories are extracted, and how much of the business (by GMV) the taxonomy actually covers.

In [ ]:
n_skus = sku_df['taxonomy_id'].nunique()
n_mapped = primary_df['product_id'].nunique()
n_categories = primary_df['master_table'].nunique()
countries = sorted(primary_df['country'].dropna().unique())
overall_coverage = gmv_df['mapped_gmv'].sum() / gmv_df['total_gmv'].sum()

print(f'Total SKUs: {n_skus:,}')
print(f'Total mapped products: {n_mapped:,}')
print(f'Categories extracted: {n_categories}')
print(f'Countries: {countries}')
print(f'Overall GMV coverage (latest month): {overall_coverage:.1%}')

In [ ]:
import matplotlib.pyplot as plt

plot_df = gmv_df.sort_values('coverage', ascending=False).copy()
plot_df['country'] = plot_df['master_table'].str.extract(r'shopee_([a-z]{2})_')
colors = plot_df['country'].map({'th': '#4C72B0', 'sg': '#DD8452'})

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(plot_df['master_table'], plot_df['coverage'] * 100, color=colors)
ax.set_ylabel('GMV coverage (%)')
ax.set_title('GMV coverage by category (latest month), TH=blue SG=orange')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
sku_counts = primary_df.groupby('master_table')['taxonomy_id'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
sku_counts.plot(kind='bar', ax=ax, color='#55A868')
ax.set_ylabel('Distinct SKUs')
ax.set_title('SKU count by category')
plt.tight_layout()
plt.show()